# Data Factory - Synthetic Dataset Generation

## 1. Setup & Configuration

In [22]:
import json
import os
import sys
from pathlib import Path
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader, UnstructuredPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq

# Setup project paths
notebook_dir = Path.cwd()
project_root = notebook_dir.parent if notebook_dir.name == 'notebooks' else notebook_dir
os.chdir(project_root)
sys.path.insert(0, str(project_root))

from src.services.llm_services import load_config

load_dotenv()
config = load_config("src/config/config.yaml")

In [23]:
# Configure Tesseract path from .env
import pytesseract
tess_path = os.getenv("TESSERACT_PATH")
if tess_path:
    # 1. Set path for pytesseract library
    pytesseract.pytesseract.tesseract_cmd = tess_path
    
    # 2. Add Tesseract directory to system PATH for unstructured's subprocess calls
    tess_dir = os.path.dirname(tess_path)
    if tess_dir not in os.environ['PATH']:
        os.environ['PATH'] = tess_dir + os.pathsep + os.environ['PATH']
    print(f"Tesseract path injected: {tess_dir}")


## 2. Data Ingestion


In [28]:
file_path = Path(config.get('source_pdf_path', './data/raw/uber.pdf'))
if not file_path.is_absolute():
    file_path = (project_root / file_path).resolve()

if not file_path.exists():
    fallback_paths = [
        (project_root / 'data/raw/uber.pdf').resolve(),
        (project_root / 'data/raw/assesment.pdf').resolve(),
    ]
    file_path = next((path for path in fallback_paths if path.exists()), file_path)

if not file_path.exists():
    raise FileNotFoundError(f"PDF not found: {file_path}")

loader = PyPDFLoader(str(file_path))
docs = loader.load()

print(f"Number of pages loaded: {len(docs)}")
if docs:
    print(f"First page metadata keys: {sorted(docs[0].metadata.keys())}")
else:
    print("No pages were loaded.")

Number of pages loaded: 142
First page metadata keys: ['creationdate', 'creator', 'moddate', 'page', 'page_label', 'producer', 'source', 'total_pages', 'trapped']


In [29]:
# Hybrid: fast text extraction + unstructured hi_res for image-only pages
import os
os.environ.setdefault("AUTO_DOWNLOAD_NLTK", "False")
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

file_path = Path(config.get('source_pdf_path', './data/raw/uber.pdf'))
if not file_path.is_absolute():
    file_path = (project_root / file_path).resolve()

if not file_path.exists():
    raise FileNotFoundError(f"PDF not found: {file_path}")

# Fast pass to get text pages
fast_pages = PyPDFLoader(str(file_path)).load()
print(f"Fast pages loaded: {len(fast_pages)}")

# Detect pages needing image/OCR processing
threshold = 200
image_pages = [i for i, d in enumerate(fast_pages) if len(d.page_content.strip()) < threshold]
print(f"Pages needing unstructured OCR/layout: {image_pages}")

# Run unstructured partition on those pages only
elements = []
if image_pages:
    try:
        from unstructured.partition.pdf import partition_pdf
    except Exception as e:
        print("unstructured.partition not available:", e)
        elements = []
    else:
        for p in image_pages:
            try:
                elems = partition_pdf(filename=str(file_path), pages=(p + 1,), strategy="hi_res", infer_table_structure=True)
            except Exception as exc:
                print(f"hi_res failed on page {p+1} ({exc.__class__.__name__}): {exc}; falling back to fast")
                try:
                    elems = partition_pdf(filename=str(file_path), pages=(p + 1,), strategy="fast")
                except Exception as exc2:
                    print(f"fast also failed on page {p+1}: {exc2}")
                    elems = []
            print(f"Extracted {len(elems)} elements from page {p+1}")
            elements.extend(elems)

print(f"Total unstructured elements extracted: {len(elements)}")
# You can now inspect `elements` or convert them to langchain Documents if needed.

Fast pages loaded: 142
Pages needing unstructured OCR/layout: [0, 76, 138, 139, 141]
hi_res failed on page 1 (IndexError): tuple index out of range; falling back to fast
Extracted 0 elements from page 1
hi_res failed on page 77 (IndexError): tuple index out of range; falling back to fast
Extracted 0 elements from page 77
hi_res failed on page 139 (IndexError): tuple index out of range; falling back to fast
Extracted 0 elements from page 139
hi_res failed on page 140 (IndexError): tuple index out of range; falling back to fast
Extracted 0 elements from page 140
hi_res failed on page 142 (IndexError): tuple index out of range; falling back to fast
Extracted 0 elements from page 142
Total unstructured elements extracted: 0
